In [1]:

# -------------------------
# Import Libraries
# -------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# -------------------------
# Load Dataset
# -------------------------

data = pd.read_csv("ner_dataset.csv", encoding="latin1")

# ============================================================
# Rename Columns
# ============================================================

data.columns = ["SentenceID", "Word", "POS", "NER"]
print(data.isnull().sum())


# ============================================================
# Fill Missing Sentence IDs
# ============================================================

data["SentenceID"] = data["SentenceID"].ffill()
data = data.dropna(subset=["Word"])
data["Word"] = data["Word"].astype(str).str.lower()
data["SentenceID"] = data["SentenceID"].astype(str)
print(data.head(15))

# ============================================================
# Convert into Sentences
# ============================================================

sentences = []

for sentence_id, group in data.groupby("SentenceID"):

    words = list(group["Word"])

    tags = list(group["POS"])

    sentences.append((words, tags))

print("Total Sentences :", len(sentences))

print("\nExample Sentence\n")

print(sentences[0])

# ============================================================
# Train Test Split
# ============================================================

train_data, test_data = train_test_split(

    sentences,

    test_size=0.2,

    random_state=42

)

print("Training Sentences :", len(train_data))

print("Testing Sentences :", len(test_data))

# ============================================================
# Build Vocabulary
# ============================================================

words = set()
tags = set()

for sentence, label in train_data:
    words.update(sentence)
    tags.update(label)

words = sorted(words)
tags = sorted(tags)

print("Vocabulary Size:", len(words))
print("Number of POS Tags:", len(tags))

print(tags)

# ============================================================
# Create Dictionaries
# ============================================================

word_to_index = {

    word: index

    for index, word in enumerate(words)

}

tag_to_index = {

    tag: index

    for index, tag in enumerate(tags)

}

index_to_tag = {

    index: tag

    for tag, index in tag_to_index.items()

}

print(tag_to_index)

# ============================================================
# Initialize Probability Matrices
# ============================================================

num_tags = len(tags)
num_words = len(words)
initial = np.ones(num_tags)

transition = np.ones((num_tags, num_tags))

emission = np.ones((num_tags, num_words))

print("Initial Matrix Shape :", initial.shape)
print("Transition Matrix Shape :", transition.shape)
print("Emission Matrix Shape :", emission.shape)

# ============================================================
# Count Frequencies
# ============================================================

for sentence, label in train_data:
    first_tag = tag_to_index[label[0]]
    initial[first_tag] += 1
    for i in range(len(label) - 1):

        current_tag = tag_to_index[label[i]]
        next_tag = tag_to_index[label[i + 1]]

        transition[current_tag][next_tag] += 1
    for word, tag in zip(sentence, label):

        word_index = word_to_index[word]
        tag_index = tag_to_index[tag]

        emission[tag_index][word_index] += 1

# ============================================================
# Normalize Probability Matrices
# ============================================================

initial = initial / initial.sum()

transition = transition / transition.sum(axis=1, keepdims=True)

emission = emission / emission.sum(axis=1, keepdims=True)

print("Initial Probabilities")
print(initial)

print("\nTransition Matrix Shape :", transition.shape)

print("Emission Matrix Shape :", emission.shape)

# ============================================================
# Convert to Log Space
# ============================================================

log_initial = np.log(initial)

log_transition = np.log(transition)

log_emission = np.log(emission)

print("Log-space conversion completed.")

# ============================================================
# Unknown Word Probability
# ============================================================

unknown_emission = np.log(np.ones(num_tags) / num_words)

# ============================================================
# Vectorized Viterbi Algorithm
# ============================================================

def viterbi(sentence):

    T = len(sentence)
    dp = np.full((num_tags, T), -np.inf)
    backpointer = np.zeros((num_tags, T), dtype=int)

    first_word = sentence[0]

    if first_word in word_to_index:
        emission_prob = log_emission[:, word_to_index[first_word]]
    else:
        emission_prob = unknown_emission

    dp[:, 0] = log_initial + emission_prob

    for t in range(1, T):

        word = sentence[t]

        if word in word_to_index:
            emission_prob = log_emission[:, word_to_index[word]]
        else:
            emission_prob = unknown_emission

        scores = dp[:, t-1][:, np.newaxis] + log_transition

        backpointer[:, t] = np.argmax(scores, axis=0)

        dp[:, t] = np.max(scores, axis=0) + emission_prob

    best_path = np.zeros(T, dtype=int)

    best_path[-1] = np.argmax(dp[:, -1])

    for t in range(T-2, -1, -1):
        best_path[t] = backpointer[best_path[t+1], t+1]

    predicted_tags = [index_to_tag[i] for i in best_path]

    return predicted_tags

# ============================================================
# Predict POS Tags
# ============================================================

true_tags = []
predicted_tags = []

for sentence, labels in test_data:

    prediction = viterbi(sentence)

    true_tags.extend(labels)

    predicted_tags.extend(prediction)

print("Prediction Completed!")

# ============================================================
# Accuracy
# ============================================================

accuracy = accuracy_score(true_tags, predicted_tags)

print("Accuracy :", accuracy)

# ============================================================
# Classification Report
# ============================================================

print(classification_report(true_tags, predicted_tags))

# ============================================================
# Test on Unseen Sentences
# ============================================================

test_sentences = [

    "I love Python programming",

    "She is reading a novel",

    "The weather is beautiful today",

    "Artificial intelligence changes the world",

    "Students are writing exams"

]

for text in test_sentences:

    sentence = text.lower().split()

    prediction = viterbi(sentence)

    print("\nSentence :", text)

    for word, tag in zip(sentence, prediction):

        print(f"{word:15} --> {tag}")

# ============================================================
# Sample Predictions
# ============================================================

for i in range(5):

    sentence, actual = test_data[i]

    predicted = viterbi(sentence)

    print("\nSentence")

    print(" ".join(sentence))

    print("\nActual Tags")

    print(actual)

    print("\nPredicted Tags")

    print(predicted)

    print("-"*60)

# ============================================================
# Final Summary
# ============================================================

print("\n==============================")
print("HMM POS Tagger Completed")
print("==============================")
print("Training Sentences :", len(train_data))
print("Testing Sentences  :", len(test_data))
print("Vocabulary Size    :", len(words))
print("Number of POS Tags :", len(tags))
print("Accuracy           :", round(accuracy * 100, 2), "%")

SentenceID    1000616
Word               10
POS                 0
NER                 0
dtype: int64
     SentenceID           Word  POS    NER
0   Sentence: 1      thousands  NNS      O
1   Sentence: 1             of   IN      O
2   Sentence: 1  demonstrators  NNS      O
3   Sentence: 1           have  VBP      O
4   Sentence: 1        marched  VBN      O
5   Sentence: 1        through   IN      O
6   Sentence: 1         london  NNP  B-geo
7   Sentence: 1             to   TO      O
8   Sentence: 1        protest   VB      O
9   Sentence: 1            the   DT      O
10  Sentence: 1            war   NN      O
11  Sentence: 1             in   IN      O
12  Sentence: 1           iraq  NNP  B-geo
13  Sentence: 1            and   CC      O
14  Sentence: 1         demand   VB      O
Total Sentences : 47959

Example Sentence

(['thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'london', 'to', 'protest', 'the', 'war', 'in', 'iraq', 'and', 'demand', 'the', 'withdrawal', 'of', '